### Create the mart layer for the Air Travel warehouse

#### Declare common variables

In [1]:
project_id = "cs329e-sp2025"
region = "us-central1"
model_name = "gemini-2.0-flash-001"
dataset = "air_travel_mrt"
region = "us-central1"

In [2]:
from google.cloud import bigquery

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset cs329e-sp2025.air_travel_mrt


#### Question 1: using the TSA traffic data, estimate how many people walk through a terminal each hour of the day and therefore how many people can a business, located in the same airport, hope to sell to? Strategy: join the tsa traffic data with the airport business data.

##### Average number of travelers per hour of the day for 12 months of the year that a business could hope to sell to

In [13]:
%%bigquery
select extract(month from t.event_date) as month, t.event_hour as hour,
b.name as business, a.name as airport, a.city, a.state, ab.terminal,
round(avg(t.passenger_count), 2) as foot_traffic
from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
on ab.business = b.name
join air_travel_int.Airport a
on ab.icao = a.icao
join air_travel_int.TSA_Traffic t
on a.icao = t.airport_icao
where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
group by extract(month from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
order by month, hour, foot_traffic desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,month,hour,business,airport,city,state,terminal,foot_traffic
0,1,0,Pints-Flights-Bites,Eppley Airfield,Omaha,NE,1,228.0
1,1,0,Pauli's Lounge,Eppley Airfield,Omaha,NE,1,228.0
2,1,0,Hudson Booksellers,Eppley Airfield,Omaha,NE,1,228.0
3,1,0,Gallery Eppley,Eppley Airfield,Omaha,NE,1,228.0
4,1,0,Hudson News/Gift,Eppley Airfield,Omaha,NE,1,228.0
5,1,0,Bud 29 Track Lounge,McCarran International Airport,Las Vegas,NV,1,39.3
6,1,0,Auntie Anne's,McCarran International Airport,Las Vegas,NV,1,39.3
7,1,0,Brighton Collectibles,McCarran International Airport,Las Vegas,NV,1,39.3
8,1,0,Alex and Ani,McCarran International Airport,Las Vegas,NV,1,39.3
9,1,0,Brookwood Farms BBQ,McCarran International Airport,Las Vegas,NV,1,39.3


##### Same query as before, but with more descriptive month and hour fields: month name and hour denoted as am/pm

In [18]:
%%bigquery
select case month
  when 1 then 'January'
  when 2 then 'February'
  when 3 then 'March'
  when 4 then 'April'
  when 5 then 'May'
  when 6 then 'June'
  when 7 then 'July'
  when 8 then 'August'
  when 9 then 'September'
  when 10 then 'October'
  when 11 then 'November'
  when 12 then 'December' end as month,
  case hour
  when 0 then '12am'
  when 1 then '1am'
  when 2 then '2am'
  when 3 then '3am'
  when 4 then '4am'
  when 5 then '5am'
  when 6 then '6am'
  when 7 then '7am'
  when 8 then '8am'
  when 9 then '9am'
  when 10 then '10am'
  when 11 then '11am'
  when 12 then '12pm'
  when 13 then '1pm'
  when 14 then '2pm'
  when 15 then '3pm'
  when 16 then '4pm'
  when 17 then '5pm'
  when 18 then '6pm'
  when 19 then '7pm'
  when 20 then '8pm'
  when 21 then '9pm'
  when 22 then '10pm'
  when 23 then '11pm' end as hour,
business, airport, city, state, terminal, foot_traffic
from
  (select extract(month from t.event_date) as month, t.event_hour as hour,
  b.name as business, a.name as airport, a.city, a.state, ab.terminal,
  round(avg(t.passenger_count), 2) as foot_traffic
  from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
  on ab.business = b.name
  join air_travel_int.Airport a
  on ab.icao = a.icao
  join air_travel_int.TSA_Traffic t
  on a.icao = t.airport_icao
  where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
  group by extract(month from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
  order by month, hour, foot_traffic desc)
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,month,hour,business,airport,city,state,terminal,foot_traffic
0,January,12am,Hudson Booksellers,Eppley Airfield,Omaha,NE,1,228.0
1,January,12am,Pauli's Lounge,Eppley Airfield,Omaha,NE,1,228.0
2,January,12am,Pints-Flights-Bites,Eppley Airfield,Omaha,NE,1,228.0
3,January,12am,Hudson News/Gift,Eppley Airfield,Omaha,NE,1,228.0
4,January,12am,Gallery Eppley,Eppley Airfield,Omaha,NE,1,228.0
5,January,12am,Brookwood Farms BBQ,McCarran International Airport,Las Vegas,NV,1,39.3
6,January,12am,Bud 29 Track Lounge,McCarran International Airport,Las Vegas,NV,1,39.3
7,January,12am,Alex and Ani,McCarran International Airport,Las Vegas,NV,1,39.3
8,January,12am,Brighton Collectibles,McCarran International Airport,Las Vegas,NV,1,39.3
9,January,12am,Auntie Anne's,McCarran International Airport,Las Vegas,NV,1,39.3


##### Create the mart table based on the previous query, while removing the limit clause

In [25]:
%%bigquery
create or replace table air_travel_mrt.airport_foot_traffic_by_month_hour as
  select case month
  when 1 then 'January'
  when 2 then 'February'
  when 3 then 'March'
  when 4 then 'April'
  when 5 then 'May'
  when 6 then 'June'
  when 7 then 'July'
  when 8 then 'August'
  when 9 then 'September'
  when 10 then 'October'
  when 11 then 'November'
  when 12 then 'December' end as month,
  case hour
  when 0 then '12am'
  when 1 then '1am'
  when 2 then '2am'
  when 3 then '3am'
  when 4 then '4am'
  when 5 then '5am'
  when 6 then '6am'
  when 7 then '7am'
  when 8 then '8am'
  when 9 then '9am'
  when 10 then '10am'
  when 11 then '11am'
  when 12 then '12pm'
  when 13 then '1pm'
  when 14 then '2pm'
  when 15 then '3pm'
  when 16 then '4pm'
  when 17 then '5pm'
  when 18 then '6pm'
  when 19 then '7pm'
  when 20 then '8pm'
  when 21 then '9pm'
  when 22 then '10pm'
  when 23 then '11pm' end as hour,
business, airport, city, state, terminal, foot_traffic
from
  (select extract(month from t.event_date) as month, t.event_hour as hour,
  b.name as business, a.name as airport, a.city, a.state, ab.terminal,
  round(avg(t.passenger_count), 2) as foot_traffic
  from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
  on ab.business = b.name
  join air_travel_int.Airport a
  on ab.icao = a.icao
  join air_travel_int.TSA_Traffic t
  on a.icao = t.airport_icao
  where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
  group by extract(month from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
  order by month, hour, foot_traffic desc)

Query is running:   0%|          |

""


##### Address the same business question, but looking at it from a week day perspective across all months and years (instead of by month seasonality)

In [24]:
%%bigquery
select extract(dayofweek from t.event_date) as weekday, t.event_hour as hour,
b.name as business, a.name as airport, a.city, a.state, ab.terminal,
round(avg(t.passenger_count), 2) as foot_traffic
from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
on ab.business = b.name
join air_travel_int.Airport a
on ab.icao = a.icao
join air_travel_int.TSA_Traffic t
on a.icao = t.airport_icao
where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
group by extract(dayofweek from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
order by weekday, hour, foot_traffic desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,weekday,hour,business,airport,city,state,terminal,foot_traffic
0,1,0,The Bluegrass Pub,Blue Grass Airport,Lexington KY,KY,1,224.0
1,1,0,The Market,Blue Grass Airport,Lexington KY,KY,1,224.0
2,1,0,Bluegrass Market,Blue Grass Airport,Lexington KY,KY,1,224.0
3,1,0,Vino,Blue Grass Airport,Lexington KY,KY,1,224.0
4,1,0,The Club at Bluegrass,Blue Grass Airport,Lexington KY,KY,1,224.0
5,1,0,Bluegrass Coffee,Blue Grass Airport,Lexington KY,KY,1,224.0
6,1,0,Hudson Vulcan,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
7,1,0,Burger King,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
8,1,0,Alabama Sport Connections,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
9,1,0,Civil Rights Trail Market,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0


##### Same query, but now with a more descriptive day of week

In [22]:
%%bigquery
select case weekday
  when 1 then 'Sunday'
  when 2 then 'Monday'
  when 3 then 'Tuesday'
  when 4 then 'Wednesday'
  when 5 then 'Thursday'
  when 6 then 'Friday'
  when 7 then 'Saturday' end as weekday,
  case hour
  when 0 then '12am'
  when 1 then '1am'
  when 2 then '2am'
  when 3 then '3am'
  when 4 then '4am'
  when 5 then '5am'
  when 6 then '6am'
  when 7 then '7am'
  when 8 then '8am'
  when 9 then '9am'
  when 10 then '10am'
  when 11 then '11am'
  when 12 then '12pm'
  when 13 then '1pm'
  when 14 then '2pm'
  when 15 then '3pm'
  when 16 then '4pm'
  when 17 then '5pm'
  when 18 then '6pm'
  when 19 then '7pm'
  when 20 then '8pm'
  when 21 then '9pm'
  when 22 then '10pm'
  when 23 then '11pm' end as hour,
  business, airport, city, state, terminal, foot_traffic
from (select extract(dayofweek from t.event_date) as weekday, t.event_hour as hour,
  b.name as business, a.name as airport, a.city, a.state, ab.terminal,
  round(avg(t.passenger_count), 2) as foot_traffic
  from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
  on ab.business = b.name
  join air_travel_int.Airport a
  on ab.icao = a.icao
  join air_travel_int.TSA_Traffic t
  on a.icao = t.airport_icao
  where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
  group by extract(dayofweek from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
  order by weekday, hour, foot_traffic desc)
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,weekday,hour,business,airport,city,state,terminal,foot_traffic
0,Sunday,12am,The Bluegrass Pub,Blue Grass Airport,Lexington KY,KY,1,224.0
1,Sunday,12am,Bluegrass Market,Blue Grass Airport,Lexington KY,KY,1,224.0
2,Sunday,12am,The Club at Bluegrass,Blue Grass Airport,Lexington KY,KY,1,224.0
3,Sunday,12am,Bluegrass Coffee,Blue Grass Airport,Lexington KY,KY,1,224.0
4,Sunday,12am,The Market,Blue Grass Airport,Lexington KY,KY,1,224.0
5,Sunday,12am,Vino,Blue Grass Airport,Lexington KY,KY,1,224.0
6,Sunday,12am,Starbucks,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
7,Sunday,12am,Bagel Bakery,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
8,Sunday,12am,Chick-fil-A,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0
9,Sunday,12am,Burger King,Birmingham-Shuttlesworth International Airport,Birmingham,AL,1,189.0


##### Create the mart table from the previous query

In [26]:
%%bigquery
create or replace table air_travel_mrt.airport_foot_traffic_by_weekday_hour as
  select case weekday
  when 1 then 'Sunday'
  when 2 then 'Monday'
  when 3 then 'Tuesday'
  when 4 then 'Wednesday'
  when 5 then 'Thursday'
  when 6 then 'Friday'
  when 7 then 'Saturday' end as weekday,
  case hour
  when 0 then '12am'
  when 1 then '1am'
  when 2 then '2am'
  when 3 then '3am'
  when 4 then '4am'
  when 5 then '5am'
  when 6 then '6am'
  when 7 then '7am'
  when 8 then '8am'
  when 9 then '9am'
  when 10 then '10am'
  when 11 then '11am'
  when 12 then '12pm'
  when 13 then '1pm'
  when 14 then '2pm'
  when 15 then '3pm'
  when 16 then '4pm'
  when 17 then '5pm'
  when 18 then '6pm'
  when 19 then '7pm'
  when 20 then '8pm'
  when 21 then '9pm'
  when 22 then '10pm'
  when 23 then '11pm' end as hour,
  business, airport, city, state, terminal, foot_traffic
from (select extract(dayofweek from t.event_date) as weekday, t.event_hour as hour,
  b.name as business, a.name as airport, a.city, a.state, ab.terminal,
  round(avg(t.passenger_count), 2) as foot_traffic
  from air_travel_int.Airport_Businesses ab join air_travel_int.Business b
  on ab.business = b.name
  join air_travel_int.Airport a
  on ab.icao = a.icao
  join air_travel_int.TSA_Traffic t
  on a.icao = t.airport_icao
  where b.name not in ('Food Court', 'Mother\'s Room', 'Conference Center')
  group by extract(dayofweek from t.event_date), t.event_hour, b.name, a.name, a.city, a.state, ab.terminal
  order by weekday, hour, foot_traffic desc)

Query is running:   0%|          |

""


#### Question 2: Using the dining data extracted from terminal maps and enriched with the LLM, where are the most common and least common food items available for purchase at various US airports around the country?

In [29]:
%%bigquery
select a.name as airport, menu_item, count(*) as count
from air_travel_int.Menu_Items m join air_travel_int.Business b
on m.business_name = b.name
join air_travel_int.Airport_Businesses ab on b.name = ab.business
join air_travel_int.Airport a on ab.icao = a.icao
group by a.name, menu_item
order by count(*) desc, airport
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airport,menu_item,count
0,San Francisco International Airport,Salads,14
1,San Francisco International Airport,Sandwiches,13
2,Austin Bergstrom International Airport,Salads,9
3,Los Angeles International Airport,Salads,9
4,Los Angeles International Airport,Sandwiches,8
5,Los Angeles International Airport,Espresso,7
6,Los Angeles International Airport,Latte,7
7,San Francisco International Airport,Burgers,7
8,Austin Bergstrom International Airport,Sandwiches,6
9,Los Angeles International Airport,Burritos,6


##### Create the mart table from the previous query

In [30]:
%%bigquery
create or replace table air_travel_mrt.airport_top_food_items as
  select a.name as airport, menu_item, count(*) as count
  from air_travel_int.Menu_Items m join air_travel_int.Business b
  on m.business_name = b.name
  join air_travel_int.Airport_Businesses ab on b.name = ab.business
  join air_travel_int.Airport a on ab.icao = a.icao
  group by a.name, menu_item
  order by count(*) desc, airport

Query is running:   0%|          |

""


#### Question 3: What are the most common categories of businesses at various US airports around the country?

In [33]:
%%bigquery
select b.category, a.name as airport, count(*) as count
from air_travel_int.Menu_Items m join air_travel_int.Business b
on m.business_name = b.name
join air_travel_int.Airport_Businesses ab on b.name = ab.business
join air_travel_int.Airport a on ab.icao = a.icao
group by b.category, a.name
order by count(*) desc, a.name
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,category,airport,count
0,Dining,Los Angeles International Airport,120
1,Dining,San Francisco International Airport,117
2,Dining,Austin Bergstrom International Airport,90
3,Dining,San Diego International Airport,75
4,Restaurant,McCarran International Airport,51
5,Dining,Fort Lauderdale Hollywood International Airport,39
6,Restaurant,Fort Lauderdale Hollywood International Airport,33
7,Dining,Bradley International Airport,30
8,Coffee Shop,Los Angeles International Airport,30
9,Dining,William P Hobby Airport,27


##### Create the mart table from the previous query

In [35]:
%%bigquery
create or replace table air_travel_mrt.top_business_categories_by_airport as
  select b.category, a.name as airport, count(*) as count
  from air_travel_int.Menu_Items m join air_travel_int.Business b
  on m.business_name = b.name
  join air_travel_int.Airport_Businesses ab on b.name = ab.business
  join air_travel_int.Airport a on ab.icao = a.icao
  group by b.category, a.name
  order by count(*) desc, a.name

Query is running:   0%|          |

""


#### Question 4: What are the highest rated and lowest rated airports in the world and in the US based on sentiment analysis done from the aiport reviews data?

##### Highest-rated airports in the whole world

In [37]:
%%bigquery
select a.name as airport_name, a.city, a.country, ar.sentiment, count(*) as num_reviews
from air_travel_int.Airport a join air_travel_int.Airport_Review ar
on a.icao = ar.icao
where ar.sentiment = 'positive'
and ar.relevant = true
group by a.name, a.city, a.country, ar.sentiment
order by count(*) desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airport_name,city,country,sentiment,num_reviews
0,Ninoy Aquino International Airport,Manila,Philippines,positive,20
1,Beja Airport / Airbase,Beja (madeira),Portugal,positive,15
2,Billy Bishop Toronto City Centre Airport,Toronto,Canada,positive,11
3,Mansfield Lahm Regional Airport,Mansfield,United States,positive,8
4,Brantford Municipal Airport,Brantford,Canada,positive,6
5,Niagara District Airport,Saint Catherines,Canada,positive,5
6,Burlington International Airport,Burlington,United States,positive,5
7,Kingston Norman Rogers Airport,Kingston,Canada,positive,5
8,Kawartha Lakes (Lindsay) Airport,Lindsay,Canada,positive,5
9,Ottawa Macdonald-Cartier International Airport,Ottawa,Canada,positive,5


##### Create the mart table from the previous query

In [38]:
%%bigquery
create or replace table air_travel_mrt.top_rated_airports_world as
  select a.name as airport_name, a.city, a.country, ar.sentiment, count(*) as num_reviews
  from air_travel_int.Airport a join air_travel_int.Airport_Review ar
  on a.icao = ar.icao
  where ar.sentiment = 'positive'
  and ar.relevant = true
  group by a.name, a.city, a.country, ar.sentiment
  order by count(*) desc

Query is running:   0%|          |

""


##### Lowest-rated airports in the whole world

In [40]:
%%bigquery
select a.name as airport_name, a.city, a.country, ar.sentiment, count(*) as num_reviews
from air_travel_int.Airport a join air_travel_int.Airport_Review ar
on a.icao = ar.icao
where ar.sentiment = 'negative'
and ar.relevant = true
group by a.name, a.city, a.country, ar.sentiment
order by count(*) desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airport_name,city,country,sentiment,num_reviews
0,Ninoy Aquino International Airport,Manila,Philippines,negative,38
1,OR Tambo International Airport,Johannesburg,South Africa,negative,30
2,Diosdado Macapagal International Airport,Angeles City,Philippines,negative,15
3,Vancouver International Airport,Vancouver,Canada,negative,11
4,Chennai International Airport,Madras,India,negative,11
5,Beja Airport / Airbase,Beja (madeira),Portugal,negative,11
6,Douala International Airport,Douala,Cameroon,negative,10
7,Montreal / Pierre Elliott Trudeau Internationa...,Montreal,Canada,negative,7
8,Lester B. Pearson International Airport,Toronto,Canada,negative,7
9,Tan Son Nhat International Airport,Ho Chi Minh City,Vietnam,negative,6


##### Create the mart table from previous query

In [41]:
%%bigquery
create or replace table air_travel_mrt.lowest_rated_airports_world as
  select a.name as airport_name, a.city, a.country, ar.sentiment, count(*) as num_reviews
  from air_travel_int.Airport a join air_travel_int.Airport_Review ar
  on a.icao = ar.icao
  where ar.sentiment = 'negative'
  and ar.relevant = true
  group by a.name, a.city, a.country, ar.sentiment
  order by count(*) desc

Query is running:   0%|          |

""


##### Highest-rated airports in the US alone

In [42]:
%%bigquery
select a.name as airport_name, a.city, a.state, ar.sentiment, count(*) as num_reviews
from air_travel_int.Airport a join air_travel_int.Airport_Review ar
on a.icao = ar.icao
where ar.sentiment = 'positive'
and ar.relevant = true
and a.country = 'United States'
group by a.name, a.city, a.state, ar.sentiment
order by count(*) desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airport_name,city,state,sentiment,num_reviews
0,Mansfield Lahm Regional Airport,Mansfield,None,positive,8
1,Teterboro Airport,Teterboro,None,positive,5
2,Burlington International Airport,Burlington,VT,positive,5
3,Hartsfield Jackson Atlanta International Airport,Atlanta,None,positive,4
4,Jack Northrop Field Hawthorne Municipal Airport,Hawthorne,None,positive,3
5,McCarran International Airport,Las Vegas,NV,positive,3
6,La Guardia Airport,New York,NY,positive,3
7,Portland International Airport,Portland,OR,positive,3
8,Capital City Airport,Harrisburg,None,positive,3
9,Reid-Hillview Airport of Santa Clara County,San Jose,None,positive,3


##### Create the mart table from the previous query

In [43]:
%%bigquery
create or replace table air_travel_mrt.top_rated_airports_us as
  select a.name as airport_name, a.city, a.state, ar.sentiment, count(*) as num_reviews
  from air_travel_int.Airport a join air_travel_int.Airport_Review ar
  on a.icao = ar.icao
  where ar.sentiment = 'positive'
  and ar.relevant = true
  and a.country = 'United States'
  group by a.name, a.city, a.state, ar.sentiment
  order by count(*) desc

Query is running:   0%|          |

""


##### Lowest-rated airports in the US

In [44]:
%%bigquery
select a.name as airport_name, a.city, a.state, ar.sentiment, count(*) as num_reviews
from air_travel_int.Airport a join air_travel_int.Airport_Review ar
on a.icao = ar.icao
where ar.sentiment = 'negative'
and ar.relevant = true
and a.country = 'United States'
group by a.name, a.city, a.state, ar.sentiment
order by count(*) desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airport_name,city,state,sentiment,num_reviews
0,Essex County Airport,Caldwell,None,negative,6
1,Laurence G Hanscom Field,Bedford,None,negative,5
2,Los Angeles International Airport,Los Angeles,CA,negative,4
3,Hazleton Municipal Airport,Hazleton,None,negative,3
4,Dallas Fort Worth International Airport,Dallas-Fort Worth,TX,negative,3
5,Washington Dulles International Airport,Washington,VA,negative,3
6,New Smyrna Beach Municipal Airport,New Smyrna Beach,None,negative,2
7,Manassas Regional Airport/Harry P. Davis Field,Manassas,None,negative,2
8,Santa Monica Municipal Airport,Santa Monica,None,negative,2
9,Capital City Airport,Harrisburg,None,negative,2


##### Create the mart table from previous query

In [45]:
%%bigquery
create or replace table air_travel_mrt.lowest_rated_airports_us as
  select a.name as airport_name, a.city, a.state, ar.sentiment, count(*) as num_reviews
  from air_travel_int.Airport a join air_travel_int.Airport_Review ar
  on a.icao = ar.icao
  where ar.sentiment = 'negative'
  and ar.relevant = true
  and a.country = 'United States'
  group by a.name, a.city, a.state, ar.sentiment
  order by count(*) desc

Query is running:   0%|          |

""


#### Question 5: Which US airlines are associated with the lowest number of flight delays and highest number of flight delays? Note: the flight delay data that we have is only available for US carriers.

##### First, look at this question from the perspective of minutes delayed (as measured by arrival time at the destination airport)

In [46]:
%%bigquery
select al.name as airline, ap.name as airport, ap.city, ap.state, sum(fd.arr_delay_min) as delay_minutes
from air_travel_int.Airport ap join air_travel_int.Flight_Delays fd
on ap.icao = fd.airport_icao
join air_travel_int.Airline al
on fd.airline_id = al.id
where fd.arr_delay_min is not null
group by al.name, ap.name, ap.city, ap.state
order by delay_minutes desc
limit 10

Query is running:   0%|          |

Downloading:   0%|          |

,airline,airport,city,state,delay_minutes
0,Delta Air Lines,Hartsfield Jackson Atlanta International Airport,Atlanta,None,40109644
1,American Airlines,Dallas Fort Worth International Airport,Dallas-Fort Worth,TX,38594115
2,Atlantic Southeast Airlines,Hartsfield Jackson Atlanta International Airport,Atlanta,None,21461111
3,United Airlines,Chicago O'Hare International Airport,Chicago,IL,20832721
4,American Eagle Airlines,Chicago O'Hare International Airport,Chicago,IL,19953115
5,American Airlines,Chicago O'Hare International Airport,Chicago,IL,19467816
6,SkyWest,Chicago O'Hare International Airport,Chicago,IL,15788771
7,American Eagle Airlines,Dallas Fort Worth International Airport,Dallas-Fort Worth,TX,15663577
8,JetBlue Airways,John F Kennedy International Airport,New York,NY,15511382
9,SkyWest,San Francisco International Airport,San Francisco,CA,14906245


##### Create the mart table from the previous query

In [47]:
%%bigquery
create or replace table air_travel_mrt.flight_delay_length_by_airline as
  select al.name as airline, ap.name as airport, ap.city, ap.state, sum(fd.arr_delay_min) as delay_minutes
  from air_travel_int.Airport ap join air_travel_int.Flight_Delays fd
  on ap.icao = fd.airport_icao
  join air_travel_int.Airline al
  on fd.airline_id = al.id
  where fd.arr_delay_min is not null
  group by al.name, ap.name, ap.city, ap.state
  order by delay_minutes desc

Query is running:   0%|          |

""


##### Second, look at this question from the perspective of delay frequency by year

In [48]:
%%bigquery
select airline, airport, city, state, round(avg(delay_frequency), 2) as delay_frequency
from
  (select al.name as airline, ap.name as airport, ap.city, ap.state, extract(year from fd.event_month) as year, count(fd.arr_delay_min) as delay_frequency
  from air_travel_int.Airport ap join air_travel_int.Flight_Delays fd
  on ap.icao = fd.airport_icao
  join air_travel_int.Airline al
  on fd.airline_id = al.id
  where fd.arr_delay_min > 0
  group by al.name, ap.name, ap.city, ap.state, extract(year from fd.event_month)
  order by delay_frequency desc)
group by airline, airport, city, state
order by delay_frequency desc
limit 12

Query is running:   0%|          |

Downloading:   0%|          |

,airline,airport,city,state,delay_frequency
0,Trans States Airlines,Pittsburgh International Airport,Pittsburgh,PA,12.00
1,Trans States Airlines,Bangor International Airport,Bangor,ME,12.00
2,Pinnacle Airlines,Delta County Airport,Escanaba,MI,12.00
3,GoJet Airlines,Norman Y. Mineta San Jose International Airport,San Jose,CA,12.00
4,Pinnacle Airlines,Chippewa County International Airport,Sault Ste Marie,MI,12.00
5,Trans States Airlines,Westchester County Airport,White Plains,NY,12.00
6,Pinnacle Airlines,Stewart International Airport,Newburgh,NY,12.00
7,Trans States Airlines,Ronald Reagan Washington National Airport,Washington,VA,12.00
8,Mesa Airlines,MBS International Airport,Saginaw,MI,12.00
9,SkyWest,Barkley Regional Airport,PADUCAH,None,11.85


##### Create the mart table from previous query

In [49]:
%%bigquery
create or replace table air_travel_mrt.flight_delay_frequency_by_airline as
  select airline, airport, city, state, round(avg(delay_frequency), 2) as delay_frequency
  from
    (select al.name as airline, ap.name as airport, ap.city, ap.state, extract(year from fd.event_month) as year, count(fd.arr_delay_min) as delay_frequency
    from air_travel_int.Airport ap join air_travel_int.Flight_Delays fd
    on ap.icao = fd.airport_icao
    join air_travel_int.Airline al
    on fd.airline_id = al.id
    where fd.arr_delay_min > 0
    group by al.name, ap.name, ap.city, ap.state, extract(year from fd.event_month)
    order by delay_frequency desc)
  group by airline, airport, city, state
  order by delay_frequency desc

Query is running:   0%|          |

""
